# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. 1.3333333333333333. isna().sum() считает пропуски, он один (None). fillna(0) заменяет пропуск нулём, получается 1, 0, 3, и среднее равно (1 + 0 + 3) / 3 = 1,33.
2. strip() убирает пробелы по краям (" a" → "a", "A " → "A"), lower() делает буквы маленькими. Получается a, a, b, то есть два разных значения.
3. df.drop_duplicates() возвращает новую таблицу, но результат нигде не сохранён. Сам df остался прежним, поэтому в нём по-прежнему три строки. Чтобы удалить дубли, нужно написать df = df.drop_duplicates(), тогда было бы 2.
4. 1005 replace(" ", "") превращает "1 000" в "1000", а to_numeric делает числа 1000 и 5. Сумма 1005.
5. [9, 10]. to_datetime превращает строки в даты, а dt.month берёт номер месяца: 30 сентября это 9, 1 октября это 10.

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [ ]:
# 1

In [2]:
import pandas as pd

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

def clean(path):
    df = pd.read_csv(path)
    print("прочитано:", len(df))
    df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
    print("после amount:", len(df))
    df["city"] = df["city"].str.strip().str.lower().map(CITY)
    print("после city:", len(df))
    df = df.drop_duplicates().copy()
    print("после дублей:", len(df))
    df["order_date"] = pd.to_datetime(df["order_date"])
    df["month"] = df["order_date"].dt.month
    print("после дат:", len(df))
    df = df[df["customer_id"] != "TEST"]
    print("после TEST:", len(df))
    assert df["amount"].isna().sum() == 0, "в amount есть NaN"
    assert df["city"].isna().sum() == 0, "в city есть NaN"
    assert df["order_id"].is_unique, "order_id не уникален"
    return df

df_clean = clean("orders.csv")
len(df_clean)

прочитано: 5463
после amount: 5463
после city: 5463
после дублей: 5283
после дат: 5283
после TEST: 5280


5280

## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [ ]:
# 2

In [6]:
df = clean("orders.csv")

paid = df["status"] == "оплачен"
sep = df["month"] == 9
octo = df["month"] == 10

прочитано: 5463
после amount: 5463
после city: 5463
после дублей: 5283
после дат: 5283
после TEST: 5280


In [5]:
s = df.loc[paid & sep, "amount"].sum()
o = df.loc[paid & octo, "amount"].sum()
print(s, o, f"{o / s - 1:.1%}")

35243410.0 34419520.0 -2.3%


In [7]:
for city in df["city"].unique():
    in_city = df["city"] == city
    s = df.loc[paid & sep & in_city, "amount"].sum()
    o = df.loc[paid & octo & in_city, "amount"].sum()
    print(city, s, o, f"{o / s - 1:.1%}")

Алматы 14501070.0 14643020.0 1.0%
Астана 9034220.0 8850530.0 -2.0%
Караганда 3648520.0 3371680.0 -7.6%
Шымкент 4558650.0 4044010.0 -11.3%
Атырау 923990.0 969860.0 5.0%
Актобе 2576960.0 2540420.0 -1.4%


#### Общее изменение −2,3% то же, что получилось на чистых данных в уроке 4, а доли городов в выручке (например, Алматы 41,15% в сентябре, Шымкент 12,93%) сходятся с числами из ДЗ 4. Шымкент, его выручка упала на 11,3%. На втором месте Караганда с −7,6%. Алматы и Атырау, наоборот, выросли.общая выручка в октябре снизилась на 2,3%, и основное падение даёт Шымкент (−11,3%), поэтому причину стоит искать там.

## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [ ]:
# 3

##### Сначала менеджер считал выручку только по оплаченным заказам, и у него получилось, что в октябре продажи упали на 22,7% по сравнению с сентябрём. Но в данных были лишние строки. Сначала я убрал тестового клиента TEST, и падение сразу сократилось до 8,2%. Потом я убрал 180 повторяющихся заказов (дублей), и падение стало 2,3%.

##### Получается, что сильнее всего цифру исказил тестовый клиент: из-за него падение выглядело почти в три раза больше, чем есть на самом деле. Дубли тоже повлияли, но меньше.

### Менеджеру можно сказать: реальное падение выручки в октябре всего 2,3%, а не 22,7%, потому что в расчёт попали тестовые заказы и дубли.

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [ ]:
# 4

In [26]:
print(df["delivery_days"].isna().sum())
print(df.loc[df["delivery_days"].isna(), "city"].value_counts())
print(df.loc[df["delivery_days"].isna(), "status"].value_counts())

158
city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64
status
оплачен    144
отменён     14
Name: count, dtype: int64


In [18]:
d = df.loc[df["city"] == "Шымкент", "delivery_days"]
print((d.dropna() <= 2).mean())           
print((d.fillna(d.median()) <= 2).mean())  
print((d.fillna(0) <= 2).mean())  

0.2984084880636605
0.28994845360824745
0.31829896907216493


#### В сроках доставки по чистым данным 158 пропусков: 144 у оплаченных заказов и 14 у отменённых. Больше всего их в Алматы (63), затем в Астане (40).

#### Доля доставки за 2 дня в Шымкенте: dropna 29,8%, fillna(медиана) 29,0%, fillna(0) 31,8%.

#### Честный вариант dropna: считаем только по заказам с известным сроком, ничего не придумываем. fillna(0) завышает долю, потому что 0 дней значит «доставили мгновенно». fillna(медиана) занижает, потому что все неизвестные заказы автоматически считаются медленными (медиана около 5 дней).

## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [ ]:
# 5b

In [20]:
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))

10 5463


In [21]:
df = pd.read_csv("orders.csv")
df = df.drop_duplicates()
df["city"] = df["city"].str.strip().str.lower().map(CITY)
print(df["city"].nunique(), len(df))

6 5283


#### В коде AI две ошибки. Первая: df.drop_duplicates() не сохраняет результат, поэтому дубли остались и строк по-прежнему 5463. Исправление: df = df.drop_duplicates(). Вторая: str.lower() не убирает пробелы по краям и не превращает латиницу (almaty, astana, shymkent) в кириллицу, поэтому вместо 6 городов получилось 10. Исправление: df["city"].str.strip().str.lower().map(CITY). После этого городов 6, а строк 5283.

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [24]:
counts = df_clean["order_date"].dt.day_name().value_counts()
print(counts)
print(counts.idxmin(), f"{counts.min() / counts.max() - 1:.1%}")

order_date
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64
Monday -13.0%


#### Больше всего заказов приходится на четверг (786), меньше всего на понедельник (684). В понедельник заказов на 13,0% меньше, чем в самый загруженный день. Воскресенье тоже слабое (697). Со вторника по субботу нагрузка почти одинаковая, от 773 до 786 заказов.

#### Для графика курьеров это значит: со вторника по субботу можно ставить поровну, а в воскресенье и понедельник можно выпускать меньше курьеров, потому что заказов там заметно меньше.

In [ ]:
# 6

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [ ]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [ ]:
# 8